In [488]:
import re
from typing import List
import time

class DailyLogActivities:
    """A Class to hold the information for a Daily Log Entry"""
    input_text: str
    processed_text: str

    def __init__(self, input_text: str):
        """Creates a new Daily Log Entry from the input_text str"""
        self.input_text = input_text
        self._getTime()
        self._getProcessedText()

    def _getTime(self):
        """Get the times from a input_text string"""
        activity_prefix = ">@"
        extra_prefix = "@<"
        time_pattern = r"(?<=\b)([0-9]{3,4}|[0-9]{1,2}:[0-9]{1,2})(?=\b)"
        matches = re.findall(time_pattern, self.input_text)
        for ind, match in enumerate(matches):
            match = match.rjust(4, "0")
            #matches[ind] = datetime.datetime.strptime(match, "%H%M")
            matches[ind] = time.strptime(match, "%H%M")
        self.start_time = matches

    def _getProcessedText(self):
        """Get the text without the times"""
        time_pattern = r"(?<=\b)[0-9]{3,4}|[0-9]{1,2}:[0-9]{1,2}(?=\b)"
        self.processed_text = "".join(re.split(time_pattern, self.input_text)).strip()

Remaining Parse Text

In [489]:
from __future__ import annotations
class ParseCurrent:
    """
    """
    def __init__(self, source_text: str):
        #self.source_text: str = source_text
        self.processed_text: str = ""
        self.unprocessed_text: str = source_text

    def getNextChars(self, next_character: int = 1) -> str:
        """
        """
        if next_character >= 1 and len(self.unprocessed_text) >= next_character + 1:
            return self.unprocessed_text[next_character]
        raise IndexError("ParseCurrent.getNextChar() there are not enough characters in unprocessed_text")

    def getCurChar(self) -> str:
        """
        """
        if len(self.unprocessed_text) >= 1:
            return self.unprocessed_text[0]
        raise IndexError("ParseCurrent.getNextChar() there are not enough characters in unprocessed_text")
    
    def getPrevChars(self, previous_character: int = -1) -> str:
        """
        """
        if previous_character <= -1 and len(self.processed_text) >=  -previous_character:
            return self.processed_text[len(self.processed_text) + previous_character]
        raise IndexError("ParseCurrent.getNextChar() there are not enough characters in processed")
    
    def itrChars(self, itr_characters: int = 1) -> None:
        """
        """
        if len(self.unprocessed_text) >= itr_characters:
            self.processed_text = "".join((self.processed_text, self.unprocessed_text[0:itr_characters]))
            self.unprocessed_text = self.unprocessed_text[itr_characters:]
            # Probably need something here to check if there's no more unprocessed characters, possibly another error
        else:
            raise IndexError("ParseCurrent.itrChar() there are no more unprocessed characters")
        
    def getChars(self, itr_characters: int = 0):
        # No match/switch statement in this python version :(
        if itr_characters == 0:
            return self.getCurChar()
        if itr_characters >= 1:
            return self.getNextChars(itr_characters)
        else:
            return self.getPrevChars(itr_characters)

    def processMatch(self, match_chars: int) -> tuple[str, ParseCurrent]:
        """
        """
        return (self.processed_text, ParseCurrent(self.unprocessed_text[match_chars:]))
        

Classes for the parse items

In [490]:
class ParseClass:
    """
    """
    def __init__(self, parse_type: str):
        self.parse_type: str = parse_type

In [491]:
import datetime

class EntryTime(ParseClass):
    """
    """
    def __init__(self, start_time: datetime.datetime):
        super().__init__(parse_type = "Time")
        self.start_time: datetime.datetime = start_time

In [492]:
class EntryActivity(ParseClass):
    """
    """
    def __init__(self, activity: str):
        super().__init__(parse_type = "Activity")
        self.activity: str = activity

In [493]:
class EntryComment(ParseClass):
    """
    """
    def __init__(self, comment: str):
        super().__init__(parse_type = "Comment")
        self.comment: str = comment

Classes for the parse search terms

In [494]:
class ParserNoMatchError(Exception):
    """
    """
    pass

In [495]:
#from __future__ import annotations
from abc import ABC, abstractmethod

class ParseSearchTerm(ABC):
    """
    """
    def __init__(self, search_term: str):
        self.search_term: str = search_term

    @abstractmethod
    def process(self, parse_current: ParseCurrent, start_pos: int) -> ParseClass:
        pass

In [496]:
class ParseSearchTermFinal(ParseSearchTerm):
    """
    """
    def __init__(self, search_term: str, return_class: ParseClass):
        super().__init__(search_term = search_term)
        self.return_class = return_class

    def process(self, parse_current: ParseCurrent, start_pos: int) -> tuple[ParseClass, int]:
        """
        """
        if parse_current.getChars(start_pos) == self.search_term:
            return (self.return_class, start_pos + 1)
        raise ParserNoMatchError("ParseSearchTermFinal.process() has no match")

In [497]:
class ParseSearchTermIntermediary(ParseSearchTerm):
    """
    """
    def __init__(self, search_term: str):
        super().__init__(search_term = search_term)
        self.next_phrases: list[ParseSearchTerm] = []

    def process(self, parse_current: ParseCurrent, start_pos: int) -> ParseClass:
        """
        """
        if parse_current.getChars(start_pos) == self.search_term:
            for ind, next_phrase in enumerate(self.next_phrases):
                try:
                    return next_phrase.process(parse_current = parse_current, start_pos = start_pos + ind + 1)
                except:
                    pass
        raise ParserNoMatchError("ParseSearchTermIntermediary.process() has no match")
    
    def addNextTerm(self, next_phrase: ParseSearchTerm):
        """
        """
        if isinstance(next_phrase, ParseSearchTerm):
            self.next_phrases.append(next_phrase)
        else:
            raise ValueError("ParseSearchTermIntermediary.addNextTerm input is not a ParseSearchTerm")

In [498]:
class ParseSearchTerms:
    """
    """
    def __init__(self):
        self.search_terms: list[ParseSearchTerm] = []

    def addSearchTerm(self, search_term: ParseSearchTerm):
        """
        """
        if isinstance(search_term, ParseSearchTerm):
            self.search_terms.append(search_term)
        else:
            raise ValueError("ParseSearchTerms.addSearchTerm input is not a ParseSearchTerm")
        
    def getSearchTerms(self, parse_current: ParseCurrent):
        for search_term in self.search_terms:
            try:
                return search_term.process(parse_current = parse_current, start_pos = 0)
            except:
                pass
        raise ParserNoMatchError("ParseSearchTerms.getSearchTerms() has no matches")

Classes for the parse search

In [499]:
class ParseSearch(ParseSearchTerm):
    """
    """
    def __init__(self, search_term: str):
        super().__init__(search_term = search_term)
        self.parser: ParseSearch = ParseSearch()

In [500]:
class Parser:
    """
    """
    def __init__(self, parser_options: ParseSearchTerms):
        self.entry_tree = []
        self.parser_options = parser_options

    def _checkMatchingCharacter(self, check_char: str):
        self.parser_options.getSearchTerms(check_char = check_char)

    def search(self, source: str):
        """
        """
        cur_mode = None
        # for char in source:
        #     print(char)
        result = ParseCurrent(source_text = source)
        return result
        

Parser Options

In [501]:
at_at_mode = ParseSearchTermFinal("@", EntryTime)
at_less_mode = ParseSearchTermFinal("<", EntryComment)
at_mode = ParseSearchTermIntermediary("@")
at_mode.addNextTerm(at_at_mode)
at_mode.addNextTerm(at_less_mode)

# greater_at_mode = ParseSearchTermFinal("@", EntryTime)
# greater_mode = ParseSearchTermIntermediary(">")
# greater_mode.addNextTerm(greater_at_mode)

parser_options = ParseSearchTerms()
#parser_options.addSearchTerm(greater_mode)
parser_options.addSearchTerm(at_mode)

Test Parser

In [502]:
test_parser = Parser(parser_options = parser_options)

#test_parser._checkMatchingCharacter("@")

test_parser.search("2026-04-24 07:29 @@ Bathroom >@ Reading Berserk @< Cool Chapter >@ @< Chapter 176")

Need to modify the check to take the entire current string and to keep iterating instead of using the same character for each next...

In [503]:
test_in = "2026-04-24 07:29 @@ Bathroom >@ Reading Berserk @< Cool Chapter >@ @< Chapter 176"

test_cur = ParseCurrent(source_text = test_in)

In [504]:
i = 0
while i < 30:
    try:
        results = parser_options.getSearchTerms(test_cur)
        break
    except:
        test_cur.itrChars(1)
    i = i + 1
print(test_cur.processed_text)
print(test_cur.unprocessed_text)
print(results[0])
matched_result = test_cur.processMatch(results[1])
print(matched_result[0])
test_cur = matched_result[1]
print(test_cur.processed_text)
print(test_cur.unprocessed_text)

2026-04-24 07:29 
@@ Bathroom >@ Reading Berserk @< Cool Chapter >@ @< Chapter 176
<class '__main__.EntryTime'>
2026-04-24 07:29 

 Bathroom >@ Reading Berserk @< Cool Chapter >@ @< Chapter 176
